# 2. Training a Model

This notebook shows how to construct a small dataset from the patch extractors and train the provided `CNNBaseline` model. The example uses the data prepared in the previous notebook. Paths are configurable so you can point to your own slides and annotations.

In [ ]:
import sys
sys.path.append("../src")
from pathlib import Path
from tvs.io.annotations import geojson_to_sqlite
from tvs.data.extractors import build_filters, make_extractors
from tvs.data.dataset import PatchDataset
from tvs.data.transforms import build_transforms
from tvs.models.cnn_baseline import CNNBaseline
from tvs.training.engine import Trainer
import torch
from torch.utils.data import DataLoader, random_split

# Paths
data_dir = Path('../tests/test_data')
slide_path = data_dir / 'test_slide.svs'
geojson_path = data_dir / 'test_slide.geojson'
db_path = data_dir / 'test_slide.db'
model_path = data_dir / 'toy_model.pt'

# Ensure annotation DB exists
geojson_to_sqlite(geojson_path, db_path, cache=False)

# Build extractors
filters = build_filters(['T', 'V'])
extractors = make_extractors(slide_path, str(db_path), filters, patch_size=512, stride=512)

label_scores = {'T': -1.0, 'V': 1.0}
full_dataset = PatchDataset(extractors, label_scores, transform=build_transforms(train=True))

# Train/validation split
train_size = max(1, int(0.8 * len(full_dataset)))
val_size = max(1, len(full_dataset) - train_size)
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
# Use validation transforms
val_ds.dataset.transform = build_transforms(train=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNBaseline().to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

trainer = Trainer(model, optim, loss_fn, device)
trainer.fit(train_loader, val_loader, epochs=1)

# Save model for later inference
model.eval()
torch.save({'model': model.state_dict()}, model_path)
print(f'Model saved to {model_path}')
